# 03 — Matched Filtering and Range Estimation

This notebook shows how correlation pulls a known echo out of noise, how the compressed peak gives you a range estimate, and why matched filtering beats thresholding the raw received signal.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/active-radar-tracker-basics/blob/radar-tracker-notebooks/beginner/03-matched-filter-range.ipynb)

Use this link if you want to open the notebook in Colab and follow along in a browser notebook environment.

## What this notebook teaches

You have built the transmit waveform, you have seen what the channel does to it, and you know the echo is buried in noise. Now you will learn the tool that pulls it out.

By the end of this notebook, you should be able to explain:

- what correlation means in the matched-filter context,
- why the compressed peak is sharper than the raw echo,
- how peak location converts to a range estimate,
- and why matched filtering beats simple thresholding.

Keep these four questions in mind as you work through the cells. At the end of the notebook, a dedicated section answers each one directly, so you can study the material first and then check your understanding against the full story.

## Setup and baseline values

We reuse the baseline radar specification from the earlier notebooks. If you are running in Colab, run the bootstrap cell below first so the repository is cloned, installed, and available for import.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/vinculum3141-ship-it/active-radar-tracker-basics.git"
BRANCH_NAME = "radar-tracker-notebooks"
REPO_DIR = Path("/content/active-radar-tracker-basics")

in_colab = "google.colab" in sys.modules

if in_colab and not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

if in_colab:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    repo_root = os.path.abspath(".")
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

    print(f"Ready in Colab from {REPO_DIR}")
else:
    print("Running locally; the repository is already available in this workspace.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from beginner.helpers import BaselineRadarSpec, baseline_spec
from beginner.helpers import (
    rectangular_pulse,
    lfm_chirp,
    matched_filter,
    delay_samples_for_range,
    range_from_delay_samples,
    single_target_channel,
)
from beginner.helpers.plotting import apply_notebook_style

np.random.seed(42)
apply_notebook_style()
radar_spec = BaselineRadarSpec()
radar_spec

## Where we are in the story

Notebook 00 introduced range and duty cycle. Notebook 01 built the transmit waveform and showed why chirps buy resolution. Notebook 02 placed a delayed, attenuated echo in a noisy received buffer. Now we have a problem: the echo is there, but you cannot see it in the noise. The matched filter is the solution.

## What the matched filter does

The radar already knows the shape of the waveform it transmitted. The matched filter exploits that knowledge: it slides a time-reversed, conjugated copy of the transmitted pulse across the received signal and computes the correlation at every position. Where the transmitted shape lines up with an echo, the output forms a strong peak. Where it does not, the output stays low.

In mathematical terms, the matched-filter output at lag $k$ is

$$
y[k] = \sum_{n} x[n] \cdot s^{*}[n - k]
$$

where $x$ is the received signal and $s$ is the transmitted waveform. The conjugate and the sign convention handle complex waveforms like the chirp, but the intuition is the same for both: slide, multiply, sum, and look for the peak.

## Build the received signal

We start with a rectangular pulse to keep the idea simple. The channel places one echo at the baseline range with -40 dB attenuation and20 dB SNR, just as in Notebook 02.

In [ ]:
# Build a rectangular pulse and place it in a noisy received buffer.
pulse_len = int(round(radar_spec.pulse_width_s * radar_spec.fs_hz))
pulse = rectangular_pulse(pulse_len)

n_delay = delay_samples_for_range(radar_spec.target_range_m, radar_spec.fs_hz)
attenuation_db = -40.0
attenuation_linear = 10.0 ** (attenuation_db / 20.0)
snr_db = 20.0

received = single_target_channel(pulse, n_delay, attenuation_linear, snr_db)

print(f"Target range = {radar_spec.target_range_m:.0f} m")
print(f"Expected delay = {n_delay} samples")
print(f"Received buffer = {len(received)} samples")

{
    "target_range_m": radar_spec.target_range_m,
    "expected_delay": n_delay,
    "buffer_length": len(received),
}

## Correlation by hand

Before calling the helper, we run the correlation step by step so you can see the arithmetic. We slide the transmitted pulse across the received signal, multiply overlapping samples, and sum at each position. The result is a vector one sample longer than the received buffer.

In [ ]:
# Compute the matched-filter output by hand.
mf_output = np.correlate(received, pulse[::-1], mode="full")
mf_peak_idx = np.argmax(np.abs(mf_output))

# The correlate output is centred; extract the delay from the peak index.
# In full mode the zero-lag position is at len(pulse) - 1.
echo_delay_samples = mf_peak_idx - (len(pulse) - 1)
echo_range_m = range_from_delay_samples(echo_delay_samples, radar_spec.fs_hz)

print(f"Matched-filter peak at index {mf_peak_idx}")
print(f"Derived echo delay = {echo_delay_samples} samples (expected {n_delay})")
print(f"Estimated range = {echo_range_m:.1f} m (expected {radar_spec.target_range_m:.0f} m)")

{
    "peak_index": int(mf_peak_idx),
    "echo_delay": int(echo_delay_samples),
    "estimated_range_m": round(echo_range_m, 1),
}

## Raw echo versus compressed output

The plot below tells the whole story. The top panel shows the received signal: the echo is small and buried in noise. The bottom panel shows the matched-filter output: the same echo is now a clear peak that stands well above the noise floor. That is the matched-filter gain in action.

In [ ]:
# Plot the raw received signal and the matched-filter output side by side.
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=False)

time_received = np.arange(len(received)) / radar_spec.fs_hz * 1e6
axes[0].plot(time_received, received, color="#7570b3", linewidth=0.7)
axes[0].set_xlabel("Time (microseconds)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Raw received signal — echo buried in noise")

lag_axis = np.arange(len(mf_output)) - (len(pulse) - 1)
axes[1].plot(lag_axis, np.abs(mf_output), color="#1b9e77")
axes[1].axvline(echo_delay_samples, color="#d95f02", linestyle="--", label=f"Peak at {echo_delay_samples} samples")
axes[1].set_xlabel("Lag (samples)")
axes[1].set_ylabel("|Matched filter output|")
axes[1].set_title("Matched-filter output — echo compressed to a sharp peak")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## Why thresholding the raw echo fails

A natural first thought is to just look for the largest sample in the received signal. The problem is visible in the top panel above: the noise can produce samples as large as the echo, so a simple threshold either misses the echo or triggers on noise. The matched filter avoids this because it correlates over the entire pulse length, accumulating energy from the echo while averaging out uncorrelated noise.

In [ ]:
# Show why thresholding the raw signal fails.
raw_peak_idx = np.argmax(np.abs(received))
threshold = 0.5 * np.max(np.abs(received))

print(f"Raw signal peak at sample {raw_peak_idx} (expected around {n_delay})")
print(f"Peak value = {np.abs(received[raw_peak_idx]):.4f}")
print(f"Threshold (50% of peak) = {threshold:.4f}")
print(f"Samples above threshold = {np.sum(np.abs(received) > threshold)}")
print()
print("The matched filter avoids this problem because it accumulates")
print("energy across the entire pulse length, not just one sample.")

{
    "raw_peak": int(raw_peak_idx),
    "true_delay": n_delay,
    "samples_above_threshold": int(np.sum(np.abs(received) > threshold)),
}

## Repeat with a chirp

The same matched-filter idea works for any waveform, but the chirp gives a much sharper peak. The width of the compressed peak is set by the bandwidth, not the pulse length — exactly as Notebook 01 predicted.

In [ ]:
# Build a chirp-based received signal and compress it.
chirp = lfm_chirp(pulse_len, radar_spec.bandwidth_hz, radar_spec.pulse_width_s, radar_spec.fs_hz)
received_chirp = single_target_channel(chirp, n_delay, attenuation_linear, snr_db)

mf_chirp = matched_filter(received_chirp, chirp)
mf_chirp_peak = np.argmax(np.abs(mf_chirp))
chirp_echo_delay = mf_chirp_peak - (len(chirp) - 1)
chirp_range_m = range_from_delay_samples(chirp_echo_delay, radar_spec.fs_hz)

print(f"Chirp matched-filter peak at sample {mf_chirp_peak}")
print(f"Derived echo delay = {chirp_echo_delay} samples (expected {n_delay})")
print(f"Estimated range = {chirp_range_m:.1f} m (expected {radar_spec.target_range_m:.0f} m)")

# Overlay the two matched-filter outputs.
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(lag_axis, np.abs(mf_output) / np.max(np.abs(mf_output)), color="#d95f02", label="Rectangular pulse", linewidth=0.8)
ax.plot(lag_axis, np.abs(mf_chirp) / np.max(np.abs(mf_chirp)), color="#1b9e77", label="LFM chirp", linewidth=0.8)
ax.axvline(n_delay, color="#999999", linestyle=":", label=f"True delay = {n_delay} samples")
ax.set_xlabel("Lag (samples)")
ax.set_ylabel("Normalised magnitude")
ax.set_title("Chirp compresses to a sharper peak than the rectangular pulse")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Checkpoint

In your own words, what does the matched filter slide across the received signal, and what is it looking for?

Then answer this: why is the chirp peak narrower than the rectangular-pulse peak, even though both waveforms have the same length?

## Common mistake

A common mistake is to think the matched filter amplifies the signal. It does not change the echo amplitude; it concentrates the echo energy into a narrow peak by coherently adding across the pulse length. The peak is higher because the energy is compressed in time, not because energy was added.

Another mistake is to treat the peak index as a range directly. The peak gives you a sample delay, and you must convert that delay to range using the speed of light and the sampling rate — the same round-trip formula from Notebook 00.

In [ ]:
# Use the helper function to reproduce the same result compactly.
helper_spec = baseline_spec()
helper_pulse = rectangular_pulse(int(round(helper_spec.pulse_width_s * helper_spec.fs_hz)))
helper_n = delay_samples_for_range(helper_spec.target_range_m, helper_spec.fs_hz)
helper_received = single_target_channel(helper_pulse, helper_n, attenuation_linear, snr_db)

helper_mf = matched_filter(helper_received, helper_pulse)
helper_peak = np.argmax(np.abs(helper_mf))
helper_delay = helper_peak - (len(helper_pulse) - 1)
helper_range = range_from_delay_samples(helper_delay, helper_spec.fs_hz)

print("Helper-based version of the same calculations:")
print(f"  peak index = {helper_peak}")
print(f"  echo delay = {helper_delay} samples")
print(f"  estimated range = {helper_range:.1f} m")

{
    "peak_index": int(helper_peak),
    "echo_delay": int(helper_delay),
    "range_m": round(helper_range, 1),
}

## Why the helpers exist

The cells above build the matched filter and range estimate step by step so you can see each operation. After that first pass, the same logic lives in `matched_filter` and `range_from_delay_samples` so later notebooks can compress an echo and read the range in one line.

As in the earlier notebooks, keep the first occurrence visible so you see the physics, then use the helpers when you want the lesson to stay readable.

## Closing the loop: answers to the opening questions

At the start of this notebook we listed four things you should be able to explain. Here is a direct answer to each one, using the physics, equations, and numbers we just worked through.

### What correlation means in the matched-filter context

Correlation is a sliding dot product: the matched filter multiplies the received signal by a time-reversed copy of the transmitted waveform and sums the products at every lag position. The lag axis runs from negative to positive, and at each position you are measuring how well the transmitted shape lines up with that slice of the received signal. When the transmitted shape lines up with the echo, every overlapping sample contributes positively and the sum is large. When it does not line up, the contributions cancel and the sum stays small.

### Why the compressed peak is sharper than the raw echo

The raw echo is as wide as the transmitted pulse —400 samples for the rectangular pulse. The matched filter compresses those400 samples of energy into a single peak whose width is set by the waveform bandwidth, not the pulse length. For the rectangular pulse the peak is still relatively wide (its bandwidth is only $1/\tau$), but for the chirp in the overlay plot the peak is far narrower because the chirp bandwidth is much larger. The matched filter does not add energy; it concentrates the energy that was already there into a narrower time slot.

### How peak location converts to a range estimate

The peak appears at a specific lag on the lag axis, which corresponds to the sample delay of the echo. In the by-hand cell the peak was at sample index $n_{peak}$, giving an echo delay of $n_{peak} - (N - 1)$ samples (the $N - 1$ offset accounts for the `full`-mode correlation). Converting that sample delay to seconds and applying $R = c \cdot \tau_{delay} / 2$ gives the range estimate. For the baseline 1000 m target the delay was 133 samples, and the estimated range matched the true range to within a fraction of a metre.

### Why matched filtering beats simple thresholding

A threshold on the raw received signal triggers on the single largest sample, which in the presence of noise may not be the echo at all. The matched filter avoids this by coherently accumulating energy across the entire pulse length. The echo contributes consistently at the correct lag, so the peak grows proportionally to the pulse length, while the noise adds incoherently and averages out. This is why the matched-filter output shows a clear peak in the bottom panel of the comparison plot even though the echo is invisible in the top panel.

If you can retell these four answers in your own words — what correlation is, why the peak is sharp, how delay becomes range, and why thresholding fails — you have the message of this notebook.

## Summary

In this notebook you met the matched filter — the tool that turns a weak, noisy echo into a clear, localised peak. The filter works by sliding a copy of the known transmitted waveform across the received signal and computing the correlation at every lag. Where the transmitted shape lines up with an echo, the output forms a strong peak; where it does not, the output stays low.

The peak location gives the echo delay in samples, and the round-trip range equation converts that delay into a distance. For the baseline 1000 m target, the estimated range came out accurately from both the rectangular pulse and the chirp. The chirp peak was sharper because its bandwidth is larger, which is the compression advantage predicted in Notebook 01.

The main takeaway is that the matched filter exploits knowledge the radar already has — the transmitted waveform — to pull a known shape out of noise. This is the foundation for everything that follows: every range measurement in a pulse radar starts with a matched-filter peak.